In [ ]:
import psutil
import pandas as pd
import numpy as np
import subprocess
import time
from datetime import datetime
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import seaborn as sns
import pickle
import joblib
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import f_oneway
from xgboost import XGBClassifier
import tensorflow as tf
from sklearn.preprocessing import scale, StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

## Monitoring system's performance

In [ ]:
# Function for monitoring performance
def session_performance_monitoring(seconds: int, label: str):
    """
    Function checks and saves various performance metrics of a device every second for a set amount of time. Given label is appended to data.
    ---
    Parameters:
    > seconds - For how long data will be collected in seconds (int).
    > label - Label that will be appended to every row (str). Has to be a value from list ['blue', 'white', 'red']
    ---
    Output:
    Pandas DataFrame with {seconds} rows and 26 columns
    ---
    
    """
    # Input validation
    if not isinstance(seconds, int) or seconds <= 0:
        raise ValueError("Invalid input of seconds parameter")
    if label not in ['blue','white','red']:
        raise ValueError("Invalid label. Accepted values: 'blue', 'white', 'red'")
        
    # Function that acquires nvidia gpu data
    def get_gpu_metrics():
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,temperature.gpu,memory.used",
                "--format=csv,noheader,nounits"
            ],
            capture_output=True,
            text=True
        )
        output = result.stdout.strip()
        gpu_util, gpu_temp, gpu_mem = output.split(", ") 
        return {
            "gpu_util_percent": float(gpu_util),
            "gpu_temp_c": float(gpu_temp)
        }
        
    # Data collection code
    data_raw = []
    for i in range(0, seconds):
        if i == 0: print(f'Starting {seconds} second ({round(seconds/60, 1)} minute) long performance monitoring session for {label} mode.\n')
            
        #Saving starting time
        start = time.time()
        
        #Timestamp
        timestamp = datetime.now().isoformat()
        #CPU Usage (%)
        cpu_usage = psutil.cpu_percent(percpu=True)
        #CPU Speed (MHz)
        cpu_speed = psutil.cpu_freq().current
        #GPU Metrics (usage and temperature)
        gpu_data = get_gpu_metrics()
        gpu_metrics = [gpu_data['gpu_util_percent'], gpu_data['gpu_temp_c']]
        #RAM Usage (%)
        ram_usage = psutil.virtual_memory().percent
        #Disk read
        disk_read = psutil.disk_io_counters().read_bytes
        #Disk write
        disk_write = psutil.disk_io_counters().write_bytes
        #Network sent
        net_sent = psutil.net_io_counters().bytes_sent
        #Network received
        net_recv = psutil.net_io_counters().bytes_recv
        
        #Appending data into one row
        current = []
        current.append(timestamp)
        current.extend(cpu_usage)
        current.append(cpu_speed)
        current.extend(gpu_metrics)                      
        current.append(ram_usage)                    
        current.append(disk_read)                 
        current.append(disk_write)                     
        current.append(net_sent)                          
        current.append(net_recv)                          
        current.append(label) 
        
        #Display current, saving data and waiting second to repeat collection
        if ((i + 1)  % 10 == 0) or i == 0: print(f'{i + 1} SECOND CHECKPOINT\n {current}\n {seconds - (i + 1)} SECONDS LEFT\n')
        if i == seconds - 1: print('Finishing session...')
        data_raw.append(current)
        time.sleep(max(0, 1 - (time.time() - start))) #Repeating function after 1s - time needed to run the code

    # Final df
    columns = [
        "timestamp",
        *[f"cpu_core_{i}_usage" for i in range(1, 17)],
        "cpu_speed_mhz",
        "gpu_usage",
        "gpu_temperature",
        "ram_usage",
        "disk_read_bytes",
        "disk_write_bytes",
        "net_bytes_sent",
        "net_bytes_recv",
        "label"
    ]

    return pd.DataFrame(data_raw, columns=columns)

In [ ]:
# Calling the function and saving results
session_data = session_performance_monitoring(20, 'red')
session_data.head()

In [ ]:
# Saving results after every session and validation
#red_2 = session_data.copy()
#white_2 = session_data.copy()
#blue_2 = session_data.copy()

#red_1 = session_data.copy()
#white_1 = session_data.copy()
#blue_1 = session_data.copy()
print(blue_1['gpu_usage'].mean(), blue_1['ram_usage'].mean(), blue_1['label'].max())
print(white_1['gpu_usage'].mean(), white_1['ram_usage'].mean(), white_1['label'].max())
print(red_1['gpu_usage'].mean(), red_1['ram_usage'].mean(), red_1['label'].max())

print(blue_2['gpu_usage'].mean(), blue_2['ram_usage'].mean(), blue_2['label'].max())
print(white_2['gpu_usage'].mean(), white_2['ram_usage'].mean(), white_2['label'].max())
print(red_2['gpu_usage'].mean(), red_2['ram_usage'].mean(), red_2['label'].max())

In [ ]:
#Loading session data
with open("blue_1.pkl", "rb") as f:
    blue_1 = pickle.load(f)
with open("white_1.pkl", "rb") as f:
    white_1 = pickle.load(f)
with open("red_1.pkl", "rb") as f:
    red_1 = pickle.load(f)

with open("blue_2.pkl", "rb") as f:
    blue_2 = pickle.load(f)
with open("white_2.pkl", "rb") as f:
    white_2 = pickle.load(f)
with open("red_2.pkl", "rb") as f:
    red_2 = pickle.load(f)

In [ ]:
blue_1.head(1)

## Data Preprocessing

In [ ]:
#Full dataset
df_0 = pd.concat([blue_1, blue_2, white_1, white_2, red_1, red_2])
df = df_0.copy()

In [ ]:
df.head(4)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['label'].value_counts()

Feature engineering

In [ ]:
# Adding average CPU usage across cores feature
cpu_core_cols = [f"cpu_core_{i}_usage" for i in range(1, 17)]
df['cpu_usage_avg'] = df[cpu_core_cols].mean(axis=1)

# Adding delta features for disk and network metrics
cols = ['disk_read_bytes', 'disk_write_bytes', 'net_bytes_sent', 'net_bytes_recv']
# Compute per-second delta
for col in cols:
    df[f'{col}_delta'] = df[col].diff().fillna(0)  # first row has no previous, so set 0

# Delete irrelevant columns
df = df.drop(columns = (['timestamp', 'disk_read_bytes', 'disk_write_bytes', 'net_bytes_sent', 'net_bytes_recv']))

df_no_rolling = df.copy() # Creating a copy without rolling mean/std features

# Adding rolling mean and standard deviation features
# Rolling window size (10 seconds)
window = 10
# Label and particular cores won't be taken into consideration
cols = ['label']
for i in range(1,17):
    cols.append(f'cpu_core_{i}_usage')
# Compute rolling mean and std
for col in df.drop(columns=cols).columns:
    df[f'{col}_10s_mean'] = df[col].rolling(window=window, min_periods=1).mean()
    df[f'{col}_10s_std'] = df[col].rolling(window=window, min_periods=1).std().fillna(0)  # first rows will be NaN, fill 0

In [ ]:
df.iloc[20:25]

In [ ]:
print(df.shape)
df.info()

In [ ]:
# Saving final dataframe
with open("df.pkl", "wb") as f:
    pickle.dump(df, f)

df.to_csv("training_dataset.csv", index=False)

## EDA

In [ ]:
# Means of selected features
features = [
    'cpu_usage_avg',   
    'gpu_usage',
    'gpu_temperature',
    'ram_usage'
]
df.groupby('label')[features].agg(['mean'])

In [ ]:
# Plots displaying distribution and comparison of various features by classes
g = sns.pairplot(
    df,
    vars=features,
    hue='label',
    palette={'blue':'blue', 'white':'orange', 'red':'red'},
    plot_kws={'alpha':0.5, 's':30}
)

for ax in g.axes.flatten():
    ax.set_xlabel(ax.get_xlabel(), fontsize=14)
    ax.set_ylabel(ax.get_ylabel(), fontsize=14)
    ax.tick_params(axis='both', labelsize=12) 

g._legend.set_title('Mode')
for text in g._legend.get_texts():
    text.set_fontsize(12)
g._legend.set_title('Mode', prop={'size':14})

plt.show()

In [ ]:
# Histograms of selected features by class
nrow = 4
ncol = 3
fig, axes = plt.subplots(nrows=nrow, ncols=ncol, figsize=(16,15))

# Function that creates a single histogram
def mode_hist(metric, label, colour, x, y, nrow):
    x_blue = df[metric].loc[df['label'] == label]
    ax = axes[x,y]
    ax.hist(x_blue, color = colour, edgecolor = 'white', bins = 12)
    ax.axvline(np.mean(x_blue), color='red', linestyle='dashed', linewidth=2, label=f'Mean: {np.mean(x_blue):.2f}')
    if x == 3: ax.set_xlabel(f"{label.capitalize()} Mode", fontsize = 14)
    if y == nrow: axes[x,y].set_ylabel(f"Freq ({metric})", fontsize = 12)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14) 
    ax.legend(fontsize = 12)

metric1 = 'cpu_usage_avg_10s_mean'
metric2 = 'gpu_usage_10s_mean'
metric3 = 'gpu_temperature_10s_mean'
metric4 = 'ram_usage_10s_mean'

mode_hist(metric1, 'blue', 'royalblue', 0, 0)
mode_hist(metric1, 'white', 'lightgrey', 0, 1)
mode_hist(metric1, 'red', 'red', 0, 2)

mode_hist(metric2, 'blue', 'royalblue', 1, 0)
mode_hist(metric2, 'white', 'lightgrey', 1, 1)
mode_hist(metric2, 'red', 'red', 1, 2)

mode_hist(metric3, 'blue', 'royalblue', 2, 0)
mode_hist(metric3, 'white', 'lightgrey', 2, 1)
mode_hist(metric3, 'red', 'red', 2, 2)

mode_hist(metric4, 'blue', 'royalblue', 3, 0)
mode_hist(metric4, 'white', 'lightgrey', 3, 1)
mode_hist(metric4, 'red', 'red', 3, 2)

plt.show()

In [ ]:
# COU speed in MHz by class
for i in df['label'].unique():
    print(f'{i}\n{df['cpu_speed_mhz'].loc[df['label']==i].value_counts()}\n')

ANOVA test

In [ ]:
df_num = df_no_rolling.drop(['label'],axis=1) # No need to check rolling mean and std features

for col in df_num.columns:
    groups = [
        df_no_rolling[df_no_rolling['label'] == r][col]
        for r in df_no_rolling['label'].unique()
    ]
    
    F_stat, p_value = f_oneway(*groups)
    print(col)
    print(f"F = {F_stat:.4f}")
    print(f"p-value = {p_value:.6f}\n")

## Developing a model

In [ ]:
# List for models' results
results = []

In [ ]:
# Dependent and explanatory variables
X = df.drop(['label'],axis=1)
y = df.label

# Creating test and training samples with test sample size = 20%
X_train, X_test, y_train, y_test = train_test_split
(
    X, y,
    test_size=0.2,
    random_state=1,
    stratify=y #Ensures equal distibution of target variable
)

### Decision Tree

For max depth = 5

In [ ]:
# Model
model_tree5 = DecisionTreeClassifier(random_state=1, max_depth=5)

# Train
model_tree5.fit(X_train, y_train)

# Predictions
y_pred = model_tree5.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results.append({'tree_5': accuracy_score(y_test, y_pred)})

In [ ]:
# Tree visualization
plt.figure(figsize=(18,8))
plot_tree(model_tree5, filled=True, feature_names=X.columns, class_names=model_tree5.classes_)
plt.show()

For max depth = 20

In [ ]:
# Model
model_tree20 = DecisionTreeClassifier(random_state=1, max_depth=20)

# Train
model_tree20.fit(X_train, y_train)

# Predictions
y_pred = model_tree20.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results.append({'tree_20': accuracy_score(y_test, y_pred)})

### Bagging

In [ ]:
# Cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=1)
# Estimator
base_estimator = DecisionTreeClassifier(random_state=1)
# Bagging classifier
model_bagging = BaggingClassifier(
    estimator=base_estimator,
    random_state=1
)
# Hyperparameters
param_grid = {
    'n_estimators': [5, 8, 10, 20, 25, 30, 40, 50, 75, 100]
}
# Grid search
grid_search = GridSearchCV(
    estimator=model_bagging,
    param_grid=param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# Best model
print("Best number of estimators:", grid_search.best_params_)

best_model_bagging = grid_search.best_estimator_

# Predictions
y_pred = best_model_bagging.predict(X_test)

# Evaluation
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results.append({'bagging': accuracy_score(y_test, y_pred)})

### Random Forest

In [ ]:
# Parameter grid
param_grid = {
    'n_estimators': [10, 20, 30, 40, 50, 100],
    'max_depth': [2, 3, 4, 5, 8, 10, 15, 20, 30],
    }

# Model
model_rf = RandomForestClassifier(random_state=1)

# Grid search with cross-validation
grid_search = GridSearchCV(
    model_rf,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Train on training data
grid_search.fit(X_train, y_train)

# Best model
best_model_rf = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

# Test set evaluation
y_pred = best_model_rf.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results.append({'random_forest': accuracy_score(y_test, y_pred)})

### XG Boost

In [ ]:
# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=1, stratify=y
)

# XGBoost classifier
model_xgb = XGBClassifier(
    eval_metric='mlogloss',
    random_state=1
)

# Parameter grid for GridSearch
param_grid = {
    'n_estimators': [50, 75, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 4, 6, 8, 10, 12]
}

# Grid search
grid_search_xgb = GridSearchCV(
    model_xgb, param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search_xgb.fit(X_train, y_train)

print("Best parameters:", grid_search_xgb.best_params_)
print("Best CV accuracy:", grid_search_xgb.best_score_)

# Evaluate on test set
y_pred = grid_search_xgb.predict(X_test)

# Map back to original labels for readability
y_pred_labels = le.inverse_transform(y_pred)
y_test_labels = le.inverse_transform(y_test)

print("Test Accuracy:", (y_pred == y_test).mean())
print("\nClassification Report:\n", classification_report(y_test_labels, y_pred_labels))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_labels, y_pred_labels))

results.append({'xgboost': (y_pred == y_test).mean()})

### Artificial Neural Network

In [ ]:
# Standarization and logarithmization of features for ANN to work properly
df_norm = df.copy()

# log transform skewed features
skew_values = df_norm.drop(['label'], axis=1).skew()
highly_skewed = skew_values[abs(skew_values) > 2].index

df_norm[highly_skewed] = df_norm[highly_skewed].apply(
    lambda x: np.log1p(x - x.min() + 1)
)

# Separate features and label
X_norm = df_norm.drop(['label'], axis=1)
y_norm = df_norm['label']

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y_norm)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_enc, test_size=0.2, random_state=1
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Cross-validation

**Cross-validation results**<br>
Neurons: 10, Hidden Layers: 1, Avg. Accuracy: 0.9583<br>
Neurons: 20, Hidden Layers: 1, Avg. Accuracy: 0.9686<br>
Neurons: 10, Hidden Layers: 2, Avg. Accuracy: 0.9714<br>
Neurons: 20, Hidden Layers: 2, Avg. Accuracy: 0.9783<br>
Neurons: 40, Hidden Layers: 2, Avg. Accuracy: 0.9837 -> best, most promising value<br>
Neurons: 10, Hidden Layers: 3, Avg. Accuracy: 0.9710<br>
Neurons: 20, Hidden Layers: 3, Avg. Accuracy: 0.9769<br>
Neurons: 40, Hidden Layers: 3, Avg. Accuracy: 0.9816<br>
Neurons: 20, Hidden Layers: 4, Avg. Accuracy: 0.9793<br>
Neurons: 20, Hidden Layers: 5, Avg. Accuracy: 0.9738<br>
Neurons: 64, Hidden Layers: 2, Avg. Accuracy: 0.9809<br>
Neurons: 64, Hidden Layers: 3, Avg. Accuracy: 0.9828 

In [ ]:
model_ann = Sequential()

# Input layer
model_ann.add(Input(shape=(X_train.shape[1],)))
# Hidden layers
model_ann.add(Dense(40, activation='relu'))
model_ann.add(Dense(40, activation='relu'))
# Output layer
model_ann.add(Dense(3, activation='softmax'))

# Final model
model_ann.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

**Result**<br>
Epoch 19: early stopping<br>
Restoring model weights from the end of the best epoch: 14.

In [ ]:
# Training ANN model
model_ann.fit(X_train, y_train, epochs=14, batch_size=32)

In [ ]:
# Predict probabilities
y_pred_prob = model_ann.predict(X_test)

# Convert probabilities -> class labels
y_pred = np.argmax(y_pred_prob, axis=1)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'[0, 1, 2] -> {le.classes_}')
print("Test Accuracy:", accuracy)

# Detailed metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

results.append({'ann': accuracy_score(y_test, y_pred)})

### Results and best model

In [ ]:
# Comparison of results
results_df = pd.DataFrame([
    {'model': k, 'accuracy': round(v, 5)} 
    for d in results 
    for k, v in d.items()
])
print(f'The best model is {str(results_df['model'].loc[results_df['accuracy'] == max(results_df['accuracy'])].values)} with accuracy of {max(results_df['accuracy'])}')
results_df

In [ ]:
# Saving the best model
# Save
joblib.dump(grid_search, "xgb_gridsearch.pkl")

print(f'Features of best model\n')

print("Best parameters:", grid_search_xgb.best_params_)
print("Best CV accuracy:", grid_search_xgb.best_score_)

# Evaluate on test set
y_pred = grid_search_xgb.predict(X_test)

# Map back to original labels for readability
y_pred_labels = le.inverse_transform(y_pred)
y_test_labels = le.inverse_transform(y_test)

print("Test Accuracy:", (y_pred == y_test).mean())
print("\nClassification Report:\n", classification_report(y_test_labels, y_pred_labels))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_labels, y_pred_labels))

#### Test without rolling features

In [ ]:
# Dependent and explanatory variables
X_no_r = df_no_rolling.drop(['label'],axis=1)
y_no_r = df_no_rolling.label

# Encode labels
le = LabelEncoder()
y_encoded_no_r = le.fit_transform(y_no_r)

# Train-test split
X_train_no_r, X_test_no_r, y_train_no_r, y_test_no_r = train_test_split(
    X_no_r, y_encoded_no_r, test_size=0.2, random_state=1, stratify=y_no_r
)

# XGBoost classifier
model_xgb_no_r = XGBClassifier(
    eval_metric='mlogloss',
    random_state=1
)

# Parameter grid for GridSearch
param_grid_no_r = {
    'n_estimators': [50, 75, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 4, 6, 8, 10, 12]
}

# Grid search
grid_search_xgb_no_r = GridSearchCV(
    model_xgb_no_r, param_grid_no_r, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search_xgb_no_r.fit(X_train_no_r, y_train_no_r)

print("Best parameters:", grid_search_xgb_no_r.best_params_)
print("Best CV accuracy:", grid_search_xgb_no_r.best_score_)

# Evaluate on test set
y_pred_no_r = grid_search_xgb_no_r.predict(X_test_no_r)

# Map back to original labels for readability
y_pred_labels_no_r = le.inverse_transform(y_pred_no_r)
y_test_labels_no_r = le.inverse_transform(y_test_no_r)

print("Test Accuracy:", (y_pred_no_r == y_test_no_r).mean())
print("\nClassification Report:\n", classification_report(y_test_labels_no_r, y_pred_labels_no_r))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_labels_no_r, y_pred_labels_no_r))

results.append({'xgboost': (y_pred_no_r == y_test_no_r).mean()})

Result: slightly worse but still very high accuracy (0.99375)

### Loading and presenting final model

In [ ]:
# Loading model and loading and adjusting data
df = pd.read_csv("training_dataset.csv", index_col = False)

X = df.drop(['label'],axis=1)
y = df.label

# Encoding target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=1,
    stratify=y #Ensures equal distibution of target variable
)

final_model = joblib.load("xgb_gridsearch.pkl")

In [ ]:
# Decoding classes (encoder uses alphabetical order so it is important to keep in mind that we got blue, red, white)
for i, class_name in enumerate(le.classes_):
    print(f"{i} --> {class_name}")

In [ ]:
# Test results of the final model
y_pred_final = final_model.predict(X_test)

y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred_final)

print("Test Accuracy:", (y_pred_final == y_test).mean())
print("\nClassification Report:\n", classification_report(y_test_labels, y_pred_labels))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_labels, y_pred_labels))

## Real-time prediction

To test model's usability in practice, a simulation has been prepared. Function presented below monitors system's performance in real time, transforms data into a desired format and based on that information predicts workload and recommended performance mode for laptop. For research purposes additional metrics and visualisations have been introduced but in practice they would slow down the function so they are not recommended for a production application.

In [ ]:
def realtime_mode_prediction(model, seconds: int, encoder=None, scaler=None, log_skewed_cols=None):
    """
    Collects system performance metrics in real time and performs ML predictions.
    Handles ANN, tree-based, or other models. Applies log and scaling if scaler is provided.

    Parameters
    ----------
    model: trained ML model with predict() method
    seconds: int, monitoring duration
    encoder: LabelEncoder, optional, for decoding target
    scaler: pre-fitted scaler, requires fitting on training data, optional (required for ANN, SVM)
    log_skewed_cols: list of column names with high skew, optional

    Returns
    -------
    pandas.DataFrame with features and predictions
    """
    import subprocess, psutil, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    # Input validation
    if not hasattr(model, "predict"):
        raise ValueError("Provided object is not a valid ML model: missing 'predict' method.")
    if not isinstance(seconds, int) or seconds <= 0:
        raise ValueError("Invalid input of seconds parameter")

    ###################################################
    # Function to acquire current performance metrics #
    ###################################################
    def current_performance():    
        # GPU metrics
        def get_gpu_metrics():
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=utilization.gpu,temperature.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True
            )
            gpu_util, gpu_temp, gpu_mem = result.stdout.strip().split(", ")
            return {"gpu_util_percent": float(gpu_util), "gpu_temp_c": float(gpu_temp)}

        cpu_usage = psutil.cpu_percent(percpu=True)
        cpu_speed = psutil.cpu_freq().current
        gpu_data = get_gpu_metrics()
        gpu_metrics = [gpu_data['gpu_util_percent'], gpu_data['gpu_temp_c']]
        ram_usage = psutil.virtual_memory().percent
        disk_read = psutil.disk_io_counters().read_bytes
        disk_write = psutil.disk_io_counters().write_bytes
        net_sent = psutil.net_io_counters().bytes_sent
        net_recv = psutil.net_io_counters().bytes_recv

        current = list(cpu_usage) + [cpu_speed] + gpu_metrics + \
                  [ram_usage, disk_read, disk_write, net_sent, net_recv]
        columns = [f"cpu_core_{i}_usage" for i in range(1, 17)] + \
                  ["cpu_speed_mhz", "gpu_usage", "gpu_temperature", "ram_usage",
                   "disk_read_bytes", "disk_write_bytes", "net_bytes_sent", "net_bytes_recv"]
        return pd.DataFrame([np.nan_to_num(current, nan=0)], columns=columns)
    ###################################################
    #       Function for data transformation          #
    ###################################################
    def data_transformation(df_history, window):
        df = df_history.copy()
        cpu_core_cols = [f"cpu_core_{i}_usage" for i in range(1, 17)]
        df['cpu_usage_avg'] = df[cpu_core_cols].mean(axis=1)
        # Computation of deltas
        delta_cols = ['disk_read_bytes', 'disk_write_bytes', 'net_bytes_sent', 'net_bytes_recv']
        for col in delta_cols:
            df[f'{col}_delta'] = df[col].diff().fillna(0)
        df = df.drop(columns=delta_cols)
        # Rolling mean and std
        for col in df.drop(columns=cpu_core_cols).columns:
            df[f'{col}_10s_mean'] = df[col].rolling(window=window, min_periods=1).mean()
            df[f'{col}_10s_std'] = df[col].rolling(window=window, min_periods=1).std().fillna(0)
        return df
    ###################################################
    #                   Final Loop                   #
    ###################################################
    final_data_list = []
    history = pd.DataFrame()
    window = 10
    time_points = []
    mode_points = []
    mode_to_num = {'blue':0,'white':1,'red':2}
    start_time = time.time()

    for i in range(seconds):
        start = time.time()
        if i == 0: print(f'Starting {seconds}s ({round(seconds/60,1)} min) session.\n')

        current = current_performance()
        history = pd.concat([history, current], ignore_index=True)
        transformed = data_transformation(history.tail(window), window)
        
        df = transformed.iloc[[-1]].copy()

        # Apply dynamic log transform if scaler is used
        if scaler is not None:
            # Identify skewed columns automatically if not provided
            if log_skewed_cols is None:
                skew_vals = df.skew()
                log_skewed_cols = skew_vals[abs(skew_vals) > 2].index.tolist()
            # Apply log
            for col in log_skewed_cols:
                df[col] = np.log1p(df[col] - df[col].min() + 1)
            # Standardize
            df[df.columns] = scaler.transform(df[df.columns])

        # Predict
        pred = model.predict(df)
        if pred.ndim > 1:
            pred = np.argmax(pred, axis=1)
        if encoder:
            pred = encoder.inverse_transform(pred)
        df['target'] = pred
        final_data_list.append(df)

        # Plot
        t = int(time.time() - start_time)
        time_points.append(t)
        mode_points.append(mode_to_num.get(pred[0], 0))

        if i % 5 == 0 or i == seconds-1:
            clear_output(wait=True)
        
            plt.figure(figsize=(20,3))
            mode_colors = ['royalblue' if m==0 else 'grey' if m==1 else 'red' for m in mode_points]
            plt.scatter(time_points, mode_points, c=mode_colors, s=80)
            plt.xlabel("Time (s)", fontsize=14, fontweight='bold')
            plt.ylabel("Mode", fontsize=14, fontweight='bold')
            plt.title("Real-Time Mode Prediction", fontsize=15, fontweight='bold')
            plt.xticks(fontsize=12)
            plt.yticks([0,1,2], ['blue','white','red'], fontsize=12)
            plt.grid(True)
        
            display(plt.gcf())
            plt.close()

        print(f"{i+1}s --> {seconds-(i+1)}s left | MODE: {pred[0]}")
        time.sleep(max(0, 1 - (time.time() - start)))

    print('Finishing session...')
    return pd.concat(final_data_list, ignore_index=True)

In [ ]:
# Setting parametres
df = pd.read_csv("training_dataset.csv", index_col = False)
y = df.label
le = LabelEncoder()
y_encoded = le.fit_transform(y)

final_model = joblib.load("final_model.pkl")

sec = 1810

In [ ]:
# Calling the function
prediction = realtime_mode_prediction(model = final_model, seconds = sec, encoder = le)
prediction.head(5)

In [ ]:
# Number of particular workload modes predicted
prediction_final = prediction.iloc[10:,:]
prediction_final['target'].value_counts()